In [1]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
df_age = pd.read_csv('ig_follower_demographics_age.csv', index_col=0)
df_city = pd.read_csv('ig_follower_demographics_city.csv', index_col=0)
df_country = pd.read_csv('ig_follower_demographics_country.csv', index_col=0)
df_gender = pd.read_csv('ig_follower_demographics_gender.csv', index_col=0)

df_city.sample(10)



,category,value
breakdown,,
city,"Bilbao, País Vasco",9
city,"Valladolid, Castilla y Leon",6
city,"Jaén, Andalusia",3
city,"Granada, Andalusia",11
city,"El Escorial, Comunidad de Madrid",2
city,"Santander, Cantabria",5
city,"Madrid, Comunidad de Madrid",255
city,"Las Rozas de Madrid, Comunidad de Madrid",2
city,"San Luis, Balearic Islands",2


In [3]:
def unificar_geographic(dfs):
    
    dfs_limpios = []

    for df in dfs:

        print(df.columns.tolist())
        
        # solo reseteamos si hace falta
        if df.index.name == "breakdown":
            df = df.reset_index()
        # renombramos columnas
        df = df.rename(columns={
            "breakdown": "dimension",
            "value": "total_followers"
        })
        # comprobamos
        print(df.columns.tolist())
        # definimos el df con los nuevos nombres de columnas y añadimos a la lista vacía
        df = df[["dimension", "category", "total_followers"]]
        dfs_limpios.append(df)
    # unimos todos los datatasets a lo largo, unos debajo de otros
    df_unico = pd.concat(dfs_limpios, ignore_index=True)
    # renombramos los valores únicos de la columna dimension con el método .map
    map_dim = {
        "city": "ciudad",
        "country": "país",
    }

    df_unico["dimension"] = df_unico["dimension"].map(map_dim)
    
    return df_unico
        
       

In [4]:
# definimos df_unico pasándole la función
dfs = [df_city, df_country]
df_unico = unificar_geographic(dfs)

['category', 'value']
['dimension', 'category', 'total_followers']
['category', 'value']
['dimension', 'category', 'total_followers']


In [5]:
df_unico.sample()

,dimension,category,total_followers
56,país,FR,5


In [6]:
def crear_df_region(df_unico):
    
    # creamos una máscara para identificar las filas con el valor 'ciudad'
    mask_ciudad = df_unico['dimension'] == 'ciudad'

    # creamos un nuevo df pasándole la máscara
    df_ciudad = df_unico.loc[mask_ciudad].copy()

    # creamos una nueva columna con el string después de la coma en la columna category
    df_ciudad['region'] = (df_ciudad['category'].str.split(',', n=1).str[1].str.strip())
    df_unico['category'] = (df_unico['category'].str.split(',', n=1).str[0].str.strip())

    # creamos un nuevo df agrupando la nueva columna con la suma de total_followers de cada region
    df_region = (df_ciudad.groupby('region', as_index=False)['total_followers'].sum())
    
    # creamos una nueva columna dimension con el valor único region
    df_region['dimension'] = 'region'
    df_region = df_region.rename(columns={'region': 'category'})
    df_region = df_region[['dimension', 'category', 'total_followers']]
    
    return df_region


In [7]:
df_region = crear_df_region(df_unico)

df_region.head()

,dimension,category,total_followers
0,region,Andalusia,44
1,region,Balearic Islands,4
2,region,Cantabria,5
3,region,Castilla y Leon,6
4,region,Cataluña,83


In [8]:
# unimos los dos df

dfs_2 = [df_unico, df_region]
df_final = pd.concat(dfs_2, ignore_index=True)

df_final.sample(10)

,dimension,category,total_followers
78,region,Islas Canarias,5
9,ciudad,Palma de Mallorca,2
42,ciudad,Cullera,2
44,ciudad,Cercedilla,2
59,país,AR,14
53,país,DO,1
45,país,DE,2
12,ciudad,Las Palmas de Gran Canaria,3
30,ciudad,Pinto,2
65,país,GB,2


In [9]:
import pandas as pd
import numpy as np
import csv
import re

def export_to_tableau(
    df: pd.DataFrame,
    path: str,
    sep: str = ";",
    encoding: str = "utf-8"
):
    """
    Limpia y exporta un DataFrame a CSV de forma segura para Tableau.

    - Elimina saltos de línea en textos (evita filas rotas)
    - Normaliza nombres de columnas
    - Controla nulos (NaN / NaT)
    - Fuerza quoting completo
    - Usa separador europeo (;)

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame limpio en pandas
    path : str
        Ruta de salida del CSV
    sep : str, optional
        Separador del CSV (default ';')
    encoding : str, optional
        Encoding del archivo (default 'utf-8')
    """

    df = df.copy()

    # 1️⃣ Normalizar nombres de columnas
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_")
          .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    # 2️⃣ Limpiar saltos de línea y retornos en columnas de texto
    cols_texto = df.select_dtypes(include="object").columns
    df[cols_texto] = df[cols_texto].replace(
        {r"[\r\n]+": " "},
        regex=True
    )

    # 3️⃣ Convertir NaN / NaT a None (Tableau-friendly)
    df = df.replace({pd.NA: None, np.nan: None})

    # 4️⃣ Exportar CSV robusto
    df.to_csv(
        path,
        sep=sep,
        index=False,
        encoding=encoding,
        quoting=csv.QUOTE_ALL
    )

    print(f"✅ CSV exportado correctamente para Tableau: {path}")
    print(f"📊 Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

In [11]:
export_to_tableau(df_final, 'geographics_IG.csv')

✅ CSV exportado correctamente para Tableau: geographics_IG.csv
📊 Filas: 85 | Columnas: 3
